In [ ]:
import pandas as pd

In [87]:
bein_src = pd.read_csv(r"S:\24.08.26\30800329_BEINDATANEWRPT.CSV", dtype='str')

In [88]:
bein = bein_src.copy()
bein['End Date'] = pd.to_datetime(bein['End Date'], dayfirst=True,errors='coerce')
bein['Customer Number'] = pd.to_numeric(bein['Customer Number']) +1014
bein['Decoder'] = pd.to_numeric(bein['Decoder']) +1014
bein['Smart Card'] = pd.to_numeric(bein['Smart Card']) +1014

In [89]:
bein['Customer Type'].unique()

array(['beIN Quartar Installment', 'beIN Bi Installment',
       'BeIN sports CC', 'CNE Subscriber', 'Hotels', 'Illegal Network',
       'Corporate Subscriber', 'Clubs', 'MCE staff (CNE staff)',
       'Head End', 'beIN Installment Sub', "In House Demo's", 'Muds',
       'Bein NC', 'Bein Companies PV', 'Clubs Free', 'Bulk DTH customer',
       'VIP_CNE', 'Temp OSN', 'beIN Dealer', 'Outsiders', 'Temp',
       'Dealers demo', 'Charge Back', 'Potential Subscriber', 'Finance'],
      dtype=object)

In [90]:
bein.columns

Index(['Customer Number', 'Customer Type', 'Entity', 'Contract Number',
       'Start Date', 'End Date', 'Plan', 'Status', 'Decoder',
       'Item Description STB', 'Smart Card', 'Item Description SC',
       'Next Billing Date', 'Billing Cycle', 'PPV Balance', 'Customer Balance',
       'Outstanding Balance'],
      dtype='object')

In [91]:
freebox = bein.copy()
freebox = freebox.loc[freebox['Plan'].str.contains('frebx',case=False)]

In [92]:
freebox = freebox.drop_duplicates(subset=['Customer Number'])

In [93]:
main_pcks = bein.copy()
main_pcks = main_pcks.loc[(main_pcks['Plan'].str.contains('prem',case=False)) 
                          | (main_pcks['Plan'].str.contains('ulti',case=False)) 
                          | (main_pcks['Plan'].str.contains('toget',case=False)) 
                          | (main_pcks['Plan'].str.contains('kick',case=False))
                          | (main_pcks['Plan'].str.contains('beIN Sports',case=False))]

In [94]:
main_grouped = main_pcks.groupby(['Customer Number']).agg(max_date = ('End Date','max')).reset_index()
main_grouped

,Customer Number,max_date
0,257720,2026-09-10
1,265710,2026-12-21
2,266186,2020-10-14
3,266230,2023-02-12
4,270904,2017-05-31
...,...,...
815560,19254291,2027-08-22
815561,19254292,2027-08-22
815562,19254293,2027-08-22
815563,19254294,2027-08-22


In [95]:
main_grouped = main_grouped.loc[main_grouped['Customer Number'].isin(freebox['Customer Number'])]

In [96]:
main_grouped

,Customer Number,max_date
40081,13834899,2027-03-28
238120,16064483,2026-03-26
757298,19184899,2025-09-11
757517,19185148,2027-08-27
757518,19185150,2026-08-31
...,...,...
767277,19196321,2026-10-07
767278,19196322,2026-10-24
767279,19196324,2025-10-06
767280,19196326,2026-10-21


In [97]:
recent_prods = pd.merge(left= main_grouped, right= main_pcks, left_on=['Customer Number','max_date'], right_on=['Customer Number','End Date'] , how = 'inner')

recent_prods = recent_prods.drop_duplicates(subset=['Customer Number','Plan'])

recent_prods = recent_prods.sort_values(['Customer Number','Plan'], ascending=[True,True])
recent_prods = recent_prods.drop_duplicates('Customer Number', keep='first')


In [98]:
recent_prods

,Customer Number,max_date,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance
0,13834899,2027-03-28,CNE Subscriber,Helio 1,4110433,29-03-2026,2027-03-28,ULTIMATE12 Ramadan26 25%off,Active,302778584.0,beIN Decoder 1000s,4.276829e+10,beIN Smartcard 1000s,29-03-2027,12M,0 Dr,0 Dr,0 DR
1,16064483,2026-03-26,CNE Subscriber,CNE Head office,3572052,27-03-2025,2026-03-26,ULTIMATE12mAdv FreBx 02.25,DIS,319334650.0,beIN Decoder 1000s,1.016179e+10,beIN Smartcard 1000s,27-03-2026,12M,0 Dr,0 Dr,0 DR
3,19184899,2025-09-11,CNE Subscriber,CNE Head office,3555224,12-03-2025,2025-09-11,PREMIUM6mEss FreBx 02.25,DIS,373046558.0,beIN 4k,1.073257e+10,CNE V7 Card,12-09-2025,6M,0 Dr,0 Dr,0 DR
4,19185148,2027-08-27,CNE Subscriber,beIN Alex Roshdy,4070618,28-02-2026,2027-08-27,PREMIUM18 Ramadan26 FWC complm.,Active,373051649.0,beIN 4k,1.073262e+10,CNE V7 Card,28-08-2027,18M,0 Dr,0 Dr,0 DR
5,19185150,2026-08-31,beIN Quartar Installment,CNE Head office,3773782,01-09-2025,2026-08-31,PREMIUM 09.24,Active,373010268.0,beIN 4k,1.073261e+10,CNE V7 Card,01-09-2026,3M,0 Dr,0 Dr,0 DR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9681,19196321,2026-10-07,CNE Subscriber,CNE Head office,3825754,08-10-2025,2026-10-07,PREMIUM 09.24,DIS,373085808.0,beIN 4k,1.073280e+10,CNE V7 Card,NaN,12M,0 Dr,0 Dr,0 DR
9683,19196322,2026-10-24,beIN Bi Installment,Meet Ghamr,3857480,25-10-2025,2026-10-24,PREMIUM 09.24,Suspended,373026919.0,beIN 4k,1.073280e+10,CNE V7 Card,25-04-2026,6M,0 Dr,0 Dr,0 DR
9684,19196324,2025-10-06,CNE Subscriber,EDD El Sayda,3587611,07-04-2025,2025-10-06,PREMIUM6mEss FreBx 02.25,DIS,373093387.0,beIN 4k,1.073276e+10,CNE V7 Card,07-10-2025,6M,0 Dr,0 Dr,0 DR
9685,19196326,2026-10-21,beIN Quartar Installment,Alex 10,3852702,22-10-2025,2026-10-21,PREMIUM 09.24,Suspended,373085595.0,beIN 4k,1.073280e+10,CNE V7 Card,22-07-2026,3M,0 Dr,0 Dr,0 DR


In [99]:
g =  recent_prods.groupby('Customer Number').agg(count = ('Customer Number','count')).reset_index()
g.loc[g['count']>1]

,Customer Number,count


In [100]:
recent_prods.loc[recent_prods['Customer Number']=='19187066']

,Customer Number,max_date,Customer Type,Entity,Contract Number,Start Date,End Date,Plan,Status,Decoder,Item Description STB,Smart Card,Item Description SC,Next Billing Date,Billing Cycle,PPV Balance,Customer Balance,Outstanding Balance


In [101]:
freebox.to_csv('freebox.csv', index=False)
recent_prods.to_csv('recent_prods.csv', index=False)

final = pd.merge(left=recent_prods, right=freebox[['Customer Number','Plan']], on='Customer Number').rename(columns={'Plan_y':'Offer', 'Plan_x':'Plan'})
final['Main pck']=''
final.loc[final['Plan'].str.contains('prem',case=False), 'Main pck'] = 'Premium'
final.loc[final['Plan'].str.contains('ulti',case=False), 'Main pck'] = 'Ultimate'
final.loc[final['Plan'].str.contains('toge',case=False), 'Main pck'] = 'Together'
final.loc[final['Plan'].str.contains('kick',case=False), 'Main pck'] = 'Kickoff'


final['Offer pck']=''
final.loc[final['Offer'].str.contains('prem',case=False), 'Offer pck'] = 'Premium'
final.loc[final['Offer'].str.contains('ulti',case=False), 'Offer pck'] = 'Ultimate'
final.loc[final['Offer'].str.contains('toge',case=False), 'Offer pck'] = 'Together'
final.loc[final['Offer'].str.contains('kick',case=False), 'Offer pck'] = 'Kickoff'


final['Offer type']=''
final.loc[final['Offer'].str.contains('ess',case=False), 'Offer type'] = 'Essential'
final.loc[final['Offer'].str.contains('vip',case=False), 'Offer type'] = 'VIP'
final.loc[final['Offer'].str.contains('adv',case=False), 'Offer type'] = 'Advanced'


final.loc[final['Status'].str.contains('DIS',case=False), 'Status'] = 'Disconnected'


In [102]:
final.to_csv('freebox final.csv', index=False)

In [103]:
final['Customer Type'].unique()

array(['CNE Subscriber', 'beIN Quartar Installment', 'Outsiders',
       'beIN Bi Installment', 'BeIN sports CC', 'Head End',
       'MCE staff (CNE staff)', 'Illegal Network', 'Corporate Subscriber',
       'Clubs', 'Bein NC', 'Bulk DTH customer'], dtype=object)

In [107]:
final.columns

Index(['Customer Number', 'max_date', 'Customer Type', 'Entity',
       'Contract Number', 'Start Date', 'End Date', 'Plan', 'Status',
       'Decoder', 'Item Description STB', 'Smart Card', 'Item Description SC',
       'Next Billing Date', 'Billing Cycle', 'PPV Balance', 'Customer Balance',
       'Outstanding Balance', 'Offer', 'Main pck', 'Offer pck', 'Offer type'],
      dtype='object')

In [108]:
full_data = bein.loc[bein['Customer Number'].isin(freebox['Customer Number'])]
full_data = full_data.sort_values(['Customer Number','Plan'], ascending=[True,True])
full_data = full_data.drop_duplicates(subset=['Customer Number','Plan'], keep='first')

In [109]:
full_data.to_csv('freebox offer full data.csv', index= False)